# 05 Models — Logistic Regression — Men's

A linear model to add structural diversity to the ensemble. Tree models and neural nets are all nonlinear and highly correlated (>0.95). Logistic regression makes fundamentally different errors, which benefits ensemble diversity even if its standalone Brier score is slightly worse.

Research supports this: Andrew Landgraf won an earlier competition with logistic regression on composite ratings, and the 2023 top solution used LR for women's.

**Inputs** (from S3 `04_preprocessing/mens/`):
- `train_features.parquet`, `stage1_features.parquet`, `stage2_features.parquet`, `feature_columns.parquet`

**Outputs** (to S3 `05_models/logistic_regression/mens/`):
- `oof_predictions.parquet`, `stage1_predictions.parquet`, `stage2_predictions.parquet`, `cv_results.parquet`

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

try:
    import optuna
except:
    !pip install optuna
    import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.isotonic import IsotonicRegression

print(f"Optuna version: {optuna.__version__}")

#### Functions

In [2]:
def read_parquet(filename):
    """Read parquet from S3 or local."""
    try:
        return pd.read_parquet(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_parquet(f"../../04_preprocessing/output/{filename}")

#### Constants

In [3]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "mens"
STAGE = "05_models/logistic_regression"
INPUT_PREFIX = f"s3://{BUCKET}/04_preprocessing/{GENDER}/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

LOCAL_OUTPUT = "output/"

#### Make output directory

In [4]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Data

In [5]:
train = read_parquet("train_features.parquet")
stage1 = read_parquet("stage1_features.parquet")
stage2 = read_parquet("stage2_features.parquet")
feature_cols = read_parquet("feature_columns.parquet")['feature'].tolist()

print(f"Training data: {train.shape}")
print(f"Stage 1 grid: {stage1.shape}")
print(f"Stage 2 grid: {stage2.shape}")
print(f"Features: {len(feature_cols)}")

Training data: (5170, 115)
Stage 1 grid: (261013, 113)
Stage 2 grid: (66430, 112)
Features: 38


## 2. Hyperparameter Tuning with Optuna

Bayesian optimization over regularization strength and penalty type.

In [ ]:
def optuna_objective(trial):
    """Optuna objective: returns mean LOGO-CV Brier score on 2003+ folds."""
    C = trial.suggest_float('C', 0.001, 100.0, log=True)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet'])
    
    if penalty == 'elasticnet':
        l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)
        solver = 'saga'
    elif penalty == 'l1':
        l1_ratio = None
        solver = 'saga'
    else:
        l1_ratio = None
        solver = 'lbfgs'
    
    tune_folds = [f for f in sorted(train['Fold'].unique()) if f >= 2003]
    train_original_tmp = train[train['TeamA'] < train['TeamB']].copy()
    
    fold_briers = []
    for fold in tune_folds:
        train_mask = (train['Fold'] != fold)
        val_mask = (train_original_tmp['Fold'] == fold)
        
        X_tr = train.loc[train_mask, feature_cols].fillna(0).values
        y_tr = train.loc[train_mask, 'Label'].values
        X_va = train_original_tmp.loc[val_mask, feature_cols].fillna(0).values
        y_va = train_original_tmp.loc[val_mask, 'Label'].values
        
        if len(X_va) == 0:
            continue
        
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_va = scaler.transform(X_va)
        
        model = LogisticRegression(
            C=C, penalty=penalty, solver=solver,
            l1_ratio=l1_ratio, max_iter=2000
        )
        model.fit(X_tr, y_tr)
        
        preds = model.predict_proba(X_va)[:, 1]
        fold_briers.append(brier_score_loss(y_va, preds))
    
    return np.mean(fold_briers)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(optuna_objective, n_trials=100, show_progress_bar=True)

print(f"\nBest trial Brier: {study.best_value:.4f}")
print(f"Best hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

## 3. LOGO Cross-Validation

In [ ]:
best = study.best_params

# Build model kwargs from best params
def make_lr_model(params):
    penalty = params['penalty']
    C = params['C']
    l1_ratio = params.get('l1_ratio', None)
    solver = 'saga' if penalty in ['l1', 'elasticnet'] else 'lbfgs'
    return LogisticRegression(C=C, penalty=penalty, solver=solver, l1_ratio=l1_ratio, max_iter=2000)

train_original = train[train['TeamA'] < train['TeamB']].copy()
folds = sorted(train['Fold'].unique())

oof_preds = []
cv_results = []

for fold in folds:
    train_mask = train['Fold'] != fold
    val_mask = (train_original['Fold'] == fold)
    
    X_train = train.loc[train_mask, feature_cols].fillna(0).values
    y_train = train.loc[train_mask, 'Label'].values
    X_val = train_original.loc[val_mask, feature_cols].fillna(0).values
    y_val = train_original.loc[val_mask, 'Label'].values
    
    if len(X_val) == 0:
        continue
    
    # Scale features per fold
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    
    model = make_lr_model(best)
    model.fit(X_train, y_train)
    
    preds = model.predict_proba(X_val)[:, 1]
    brier = brier_score_loss(y_val, preds)
    
    cv_results.append({
        'Fold': fold,
        'BrierScore': brier,
        'Games': len(y_val),
        'BestRound': 1  # single fit, no boosting
    })
    
    fold_oof = train_original.loc[val_mask, ['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val']].copy()
    fold_oof['Pred'] = preds
    oof_preds.append(fold_oof)
    
    print(f"  Fold {fold}: Brier={brier:.4f}, Games={len(y_val)}")

oof_df = pd.concat(oof_preds, ignore_index=True)
cv_df = pd.DataFrame(cv_results)

In [8]:
overall_brier = brier_score_loss(oof_df['Label'], oof_df['Pred'])
stage1_oof = oof_df[oof_df['IsStage1Val'] == 1]
stage1_brier = brier_score_loss(stage1_oof['Label'], stage1_oof['Pred']) if len(stage1_oof) > 0 else None

print(f"\nOverall OOF Brier Score: {overall_brier:.4f}")
print(f"Stage 1 (2022-2025) Brier Score: {stage1_brier:.4f}" if stage1_brier else "No Stage 1 data")
print(f"Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")


Overall OOF Brier Score: 0.1855
Stage 1 (2022-2025) Brier Score: 0.1972
Mean Brier: 0.1854 +/- 0.0186


## 4. Train Final Model and Calibrate

In [ ]:
# Train final model on all data
X_all = train[feature_cols].fillna(0).values
y_all = train['Label'].values

final_scaler = StandardScaler()
X_all_scaled = final_scaler.fit_transform(X_all)

final_model = make_lr_model(best)
final_model.fit(X_all_scaled, y_all)

# Calibration
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(oof_df['Pred'].values, oof_df['Label'].values)

oof_df['PredCalibrated'] = calibrator.predict(oof_df['Pred'].values)
calibrated_brier = brier_score_loss(oof_df['Label'], oof_df['PredCalibrated'])
print(f"OOF Brier (raw): {overall_brier:.4f}")
print(f"OOF Brier (calibrated): {calibrated_brier:.4f}")

## 5. Generate Predictions

In [10]:
# Stage 1
X_s1 = final_scaler.transform(stage1[feature_cols].fillna(0).values)
stage1['Pred_logistic_regression'] = final_model.predict_proba(X_s1)[:, 1]
stage1['Pred_logistic_regression_calibrated'] = calibrator.predict(
    stage1['Pred_logistic_regression'].values
).clip(0.02, 0.98)

# Stage 2
X_s2 = final_scaler.transform(stage2[feature_cols].fillna(0).values)
stage2['Pred_logistic_regression'] = final_model.predict_proba(X_s2)[:, 1]
stage2['Pred_logistic_regression_calibrated'] = calibrator.predict(
    stage2['Pred_logistic_regression'].values
).clip(0.02, 0.98)

print(f"Stage 1 predictions range: [{stage1['Pred_logistic_regression_calibrated'].min():.3f}, {stage1['Pred_logistic_regression_calibrated'].max():.3f}]")
print(f"Stage 2 predictions range: [{stage2['Pred_logistic_regression_calibrated'].min():.3f}, {stage2['Pred_logistic_regression_calibrated'].max():.3f}]")

# Evaluate on actual Stage 1 games
stage1_actual = stage1.dropna(subset=['Label'])
if len(stage1_actual) > 0:
    s1_brier = brier_score_loss(stage1_actual['Label'], stage1_actual['Pred_logistic_regression_calibrated'])
    print(f"Stage 1 Brier (calibrated): {s1_brier:.4f}")

Stage 1 predictions range: [0.020, 0.980]
Stage 2 predictions range: [0.020, 0.980]
Stage 1 Brier (calibrated): 0.1916


## 6. Feature Coefficients

Unlike tree models, logistic regression gives us interpretable coefficients. Positive = favors TeamA winning when TeamA has a higher value.

In [11]:
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': final_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print("Feature Coefficients (sorted by absolute value):")
for _, row in coef_df.iterrows():
    print(f"  {row['Feature']:30s}: {row['Coefficient']:+.4f}")

Feature Coefficients (sorted by absolute value):
  AvgOrdinalRankDiff            : +0.9554
  EloDiff                       : +0.7775
  AvgPointDiffDiff              : +0.7288
  TopSystemsAvgRankDiff         : -0.7051
  WLKDiff                       : -0.4858
  POMDiff                       : +0.3757
  SOSDiff                       : +0.2768
  RecentFGPctDiff               : +0.2196
  RecentOffEffDiff              : -0.1905
  WinPctDiff                    : -0.1885
  IsPowerConfDiff               : +0.1722
  AvgORDiff                     : +0.1548
  RecentAvgPointDiffDiff        : -0.1437
  SAGDiff                       : +0.1390
  SeedDiff                      : -0.1153
  AvgTODiff                     : -0.1103
  FTPctDiff                     : +0.1075
  RecentNetEffDiff              : -0.1059
  ScoreStdDiff                  : -0.1058
  SeedB                         : +0.0935
  SeedA                         : -0.0935
  AvgDRDiff                     : -0.0921
  AvgAstDiff               

## 7. Save Outputs

In [12]:
outputs = {
    'oof_predictions': oof_df[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val', 'Pred', 'PredCalibrated']],
    'stage1_predictions': stage1[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Pred_logistic_regression', 'Pred_logistic_regression_calibrated']],
    'stage2_predictions': stage2[['ID', 'Season', 'TeamA', 'TeamB', 'Pred_logistic_regression', 'Pred_logistic_regression_calibrated']],
    'cv_results': cv_df,
}

for name, df in outputs.items():
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

Saved to S3: s3://march-machine-learning-mania-2026/05_models/logistic_regression/mens/oof_predictions.parquet ((2585, 9))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/logistic_regression/mens/stage1_predictions.parquet ((261013, 7))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/logistic_regression/mens/stage2_predictions.parquet ((66430, 6))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/logistic_regression/mens/cv_results.parquet ((40, 4))


## 8. Summary

In [13]:
print("=" * 60)
print("LOGISTIC REGRESSION MODEL SUMMARY — MEN'S")
print("=" * 60)
print(f"\nOOF Brier Score (raw): {overall_brier:.4f}")
print(f"OOF Brier Score (calibrated): {calibrated_brier:.4f}")
print(f"CV Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")
print(f"\nTop 5 features (by |coefficient|):")
for _, row in coef_df.head().iterrows():
    print(f"  {row['Feature']}: {row['Coefficient']:+.4f}")

LOGISTIC REGRESSION MODEL SUMMARY — MEN'S

OOF Brier Score (raw): 0.1855
OOF Brier Score (calibrated): 0.1823
CV Mean Brier: 0.1854 +/- 0.0186

Top 5 features (by |coefficient|):
  AvgOrdinalRankDiff: +0.9554
  EloDiff: +0.7775
  AvgPointDiffDiff: +0.7288
  TopSystemsAvgRankDiff: -0.7051
  WLKDiff: -0.4858
